# RunPod: the variant `flat_h1_rollout4` (V1) against F1

The rollout model: history 1 and a 4-step open-loop loss (`level1.wm.history_size=1`, `level1.wm.rollout_n=4`), F1's settings otherwise. The questions: does the planner's history 1 suffice (it plans with history 1), and do rollouts bring back what the latent loses (the view, the faced tile).

Each variant has its own notebook and its own pod (A100, about $1.8/h): two variants train in parallel. Once per pod, in a terminal: `git clone https://github.com/quastAI/AAC.git /root/AAC`, then open this notebook from there. The paths (what lives on the network volume and what on the local disk) and the helpers are in `notebooks/pod.py`, shared with `runpod_flat.ipynb`.

**Before the variants**: `runpod_flat.ipynb` has run the probe suite on F1 and F3 (F1 and the random baseline on task set A) and uploaded them. This notebook takes them from the Hub; what it does not find there it runs itself (the same jobs, skipped when done), which costs this pod the time.

| step | what |
|---|---|
| 0 | configuration, Hugging Face login, the PyTorch environment, the dataset from the Hub |
| 1 | F1 and its results on set A from the Hub, a **smoke run** of this variant on the real files (a few batches, then the probes) |
| 2 | the run of `experiments/run.py run V1 --run flat_h1_rollout4`, detached: training (10 epochs, about 3.3 hours; resumes after a pod restart), the probe suite at its end, the cheap P1 on set A, the comparison with F1 and the random baseline (`reports/V1_flat_h1_rollout4_A_val.md`, `..._models.md`) |
| 3 | this run, F1's and the baseline's results and the reports to the Hub |

A step that finished is skipped when you run its cell again.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
import os, shutil, sys, time
from pathlib import Path

EXPERIMENT, RUN = "V1", "flat_h1_rollout4"   # a trained run of the experiment in experiments/experiments.yaml
REFERENCE = "flat"                          # F1, the run this variant is compared with (a reference run of the experiment)

REPO = Path("/root/AAC")                    # the git clone, on the local disk (pod.py has the other paths and why they are where they are)
sys.path.insert(0, str(REPO / "notebooks"))
import pod

HF_REPO_ID = "quastAI/craftax-classic-pqn-rnn-v1"   # the dataset repo of runpod_dataset.ipynb (data/), runs go to runs/
HF_PRIVATE = True
SEEDS = "42"                                # one seed per variant (settled on 2026-10-20)
MAX_EPOCHS = 10                             # as F1: 155k steps, about 3.3 hours on an A100; the cosine schedule ends here
USE_WANDB = True                            # live curves in Weights & Biases (project hjepa-flat); False = the metrics CSV only

In [ ]:
# --- 0. Paths, Hugging Face login, the PyTorch environment (installed again on a new pod) and the dataset ----------------
pod.configure(repo=REPO, seeds=SEEDS)
from pod import DATA, H_JEPA, PYTHON, RUNNER, launch, progress, run_dir, running, sh, show, wait

TRAIN_OVERRIDES = pod.train_overrides(USE_WANDB)   # Hydra overrides for every training run
api = pod.login(HF_REPO_ID, HF_PRIVATE, USE_WANDB)
pod.install(api, HF_REPO_ID, USE_WANDB)
SPEC = pod.run_spec(EXPERIMENT, RUN)               # the run's training config and overrides
print(RUN, SPEC["config"], *SPEC.get("overrides", []))

## 1. F1 from the Hub, a smoke run

F1's folder (the model, its reports, its planning results on set A) and the random baseline on set A come from `runs/` of the Hub repo. A probe report of an older suite is moved aside (`health/probe_suite_v1/`), so that F1 is probed again on this pod. The smoke run trains this variant for 20 batches with its own overrides and runs the probe suite on 16 clips: a wrong override or a crash in the variant's code path shows here, not after hours.

In [ ]:
# --- 1. F1 and its results on set A from the Hub (what this pod's volume has is kept) --------------------------------------------
pod.fetch(api, HF_REPO_ID, f"{REFERENCE}/seed{SEEDS.split(',')[0]}")   # the model, its reports and planning results
pod.fetch(api, HF_REPO_ID, "random/eval_A_val", required=False)        # the random baseline on set A
reference = run_dir(REFERENCE)
planned = (reference / "eval_A_val_craftax_plan" / "planning_report.md").exists()
print("F1 on set A:", "done" if planned else "missing, planned on this pod (F3 in runpod_flat.ipynb plans it once for all variants)")
print("F1's probe report:", "current" if pod.current_probe_report(reference) else "missing or older, probed again on this pod")

In [ ]:
# --- 1. Smoke run of this variant on the real files (the full output is in LOGS/smoke.log) ----------------------------------
SMOKE = f"{RUN}_smoke"
shutil.rmtree(run_dir(SMOKE), ignore_errors=True)
t0 = time.time()
sh([PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *SPEC.get("overrides", []), f"output_model_name={SMOKE}",
    f"seed={SEEDS.split(',')[0]}", "trainer.max_epochs=1", "+trainer.limit_train_batches=20", "+trainer.limit_val_batches=2",
    "health.probe_clips=16", *TRAIN_OVERRIDES, "wandb.enabled=false"], cwd=H_JEPA, quiet=True, log="smoke")   # no W&B run for a smoke run
out = run_dir(SMOKE)
assert (out / f"{SMOKE}_object.ckpt").exists(), "the smoke run wrote no model"
assert (out / "health" / "report.md").exists() and (out / "health" / "probe_report.md").exists(), "the smoke run wrote no reports"
print(f"smoke run ok in {time.time() - t0:.0f} s: {out}")

## 2. Training, probe suite, cheap P1 against F1

One detached job chain (`run.py run EXPERIMENT --run RUN`, the names of the configuration cell): the task set A, the training (the probe suite runs at its end), F1 and the random baseline on set A when they are missing, this run on set A, then the two comparison reports. If the pod restarts, run the start cell again: the training resumes from `lightning_resume/last.ckpt`, the finished jobs are skipped. Two epochs in a row with a critical finding stop the training (`health.stop_after_critical_epochs`); the model is still saved and the jobs after it run, so read `report.md` first.

In [ ]:
# --- 2. Start (or resume) the run, detached ----------------------------------------------------------------------------------
OVERRIDES = [f"trainer.max_epochs={MAX_EPOCHS}", *TRAIN_OVERRIDES]
sh([*RUNNER, "plan", EXPERIMENT, "--run", RUN, "--seeds", SEEDS, "--override", *OVERRIDES])
launch(RUN, [*RUNNER, "run", EXPERIMENT, "--run", RUN, "--mode", "local", "--seeds", SEEDS, "--override", *OVERRIDES])

In [ ]:
# --- 2. Monitor (run as often as you like) -----------------------------------------------------------------------------------------
print(RUN, "RUNNING" if running(RUN) else "not running")
print("\n".join(progress(RUN, "run  ", 1)))   # the job that runs now
print("\n".join(progress(RUN, "Epoch", 2) + progress(RUN, "[health]", 3) + progress(RUN, "[probe suite]", 2) + progress(RUN, "success_rate", 2)))
for p in sorted((run_dir(RUN) / "health").glob("epoch_*.json"))[-1:]:
    print("last epoch report:", p.name)

In [ ]:
# --- 2. Loss curves (run as often as you like, also while it trains: the metrics CSV is written as it goes) ----------------------
from IPython.display import Image, display
sh([PYTHON, REPO / "analysis" / "plot_training.py", "--run", run_dir(RUN)])
display(Image(filename=str(run_dir(RUN) / "health" / "loss_curves.png")))

In [ ]:
# --- 2. Wait for the chain, then the reports: training, probe suite, the comparisons with F1 ------------------------------------------
wait(RUN, "Epoch")
show(run_dir(RUN) / "health" / "report.md", 40)
show(run_dir(RUN) / "health" / "probe_report.md", 120)
for report in sorted((DATA / "reports").glob(f"{EXPERIMENT}_{RUN}_*.md")):
    print("\n=====", report.name)
    show(report, 80)

## 3. Upload

This run, F1's folder (planning results or a probe report made on this pod) and the random baseline go to `runs/`, this variant's reports to `reports/` of the Hub repo. Left out: the resumable training state, the per-epoch models and the smoke run.

In [ ]:
# --- 3. Runs and reports to the Hub --------------------------------------------------------------------------------------------------------
pod.upload(api, HF_REPO_ID, folders=[RUN, REFERENCE, "random"], reports=[f"{EXPERIMENT}_{RUN}_*.md"])